In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
assert torch.cuda.is_available()
%cd /mlx/users/zongyu.yin/playground/samantha

sample_rate = 24000
hop_length = 240
batch_size = 2
shuffle_buffer_size = 10
num_workers = 2

In [ ]:
import librosa
import matplotlib.pyplot as plt
from librosa.feature.inverse import mel_to_audio
from torchaudio.functional import DB_to_amplitude

def plot_waveform(waveform, sr, title="Waveform", ax=None):
    waveform = waveform.numpy()

    num_channels, num_frames = waveform.shape
    time_axis = torch.arange(0, num_frames) / sr

    if ax is None:
        _, ax = plt.subplots(num_channels, 1)
    ax.plot(time_axis, waveform[0], linewidth=1)
    ax.grid(True)
    ax.set_xlim([0, time_axis[-1]])
    ax.set_title(title)


def plot_spectrogram(specgram, title=None, ylabel="freq_bin", ax=None):
    if ax is None:
        _, ax = plt.subplots(1, 1)
    if title is not None:
        ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.imshow(specgram, origin="lower", aspect="auto", interpolation="nearest")

def mel2audio(mel):
    mel_db2amp = DB_to_amplitude(mel, ref=1, power=1)
    audio = mel_to_audio(mel_db2amp.numpy(), sr=sample_rate, n_fft=512, win_length=400, hop_length=hop_length, n_iter=100)
    return audio
    

# Dataloader

In [ ]:
from recipes.datasets.mcc.mix import VocalWebDataModule

pl_datamodule = VocalWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
)

val_loaders = pl_datamodule.val_dataloader()
vocal_loader, speech_loader = val_loaders[0], val_loaders[1]

In [ ]:
vocal_iter, speech_iter = iter(vocal_loader), iter(speech_loader)

In [ ]:
vocal_batch, speech_batch = next(vocal_iter), next(speech_iter)

In [ ]:
print("=======Vocal=======")
for text, audio in zip(vocal_batch["text"], vocal_batch["audio"]):
    print(text)
    ipd.display(ipd.Audio(audio, rate=sample_rate))
print("=======Speech=======")
for text, audio in zip(speech_batch["text"], speech_batch["audio"]):
    print(text)
    ipd.display(ipd.Audio(audio, rate=sample_rate))

# CTC

## Load ckpt and model

In [ ]:
import os
import samantha.utils.hdfs_helper as hh
from recipes.umm_062.modules.lit_module import Stage2
from transformers import BertTokenizer

ckpt_path = "/mnt/bn/zongyu-lq/logs/umm/stage2_music/checkpoints/step=0010000-loss=0.732.ckpt"
stage_2 = Stage2.load_from_checkpoint(ckpt_path).to("cuda").eval()
stage_2.tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')

In [ ]:
import os
import samantha.utils.hdfs_helper as hh
from recipes.umm_062.modules.lit_module import Stage3
from transformers import BertTokenizer

ckpt_path = "/mnt/bn/zongyu-lq/logs/umm/stage3_music_vq32768x32/checkpoints/step=0005000-loss=0.946.ckpt"
stage_3 = Stage3.load_from_checkpoint(ckpt_path).to("cuda").eval()
stage_3.tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')

In [ ]:
umm_model = stage_3

## Get melspec

In [ ]:
print("=======Vocal=======")
model_input = {"audio": vocal_batch["audio"].to("cuda"), "text": vocal_batch["text"]}
recon_mel, gt_mel = umm_model.get_mel(model_input).values()
mean = umm_model.model.audio_transform.mean
std = umm_model.model.audio_transform.std
recon_mel = (recon_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
gt_mel = (gt_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()

for audio, recon_m, gt_m in zip(vocal_batch["audio"], recon_mel, gt_mel):
    print("-----------------------")
    recon_audio = mel2audio(recon_m)
    gt_audio = mel2audio(gt_m)
    fig, axs = plt.subplots(3, 1)
    print("Groud Truth")
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("Groud Truth (mel -> audio)")
    ipd.display(ipd.Audio(gt_audio, rate=sample_rate))
    print("Reconstruction (mel -> audio)")
    ipd.display(ipd.Audio(recon_audio, rate=sample_rate))
    plot_waveform(audio, sample_rate, title="Original waveform", ax=axs[0])
    plot_spectrogram(gt_m, title="GT", ax=axs[1])
    plot_spectrogram(recon_m, title="Recon", ax=axs[2])
    fig.tight_layout()

In [ ]:
print("=======Speech=======")
model_input = {"audio": speech_batch["audio"].to("cuda"), "text": speech_batch["text"]}
recon_mel, gt_mel = umm_model.get_mel(model_input).values()
mean = umm_model.model.audio_transform.mean
std = umm_model.model.audio_transform.std
recon_mel = (recon_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()
gt_mel = (gt_mel.detach().transpose(1, 2) * std + mean).transpose(1, 2).cpu()

for audio, recon_m, gt_m in zip(speech_batch["audio"], recon_mel, gt_mel):
    print("-----------------------")
    recon_audio = mel2audio(recon_m)
    gt_audio = mel2audio(gt_m)
    fig, axs = plt.subplots(3, 1)
    print("Groud Truth")
    ipd.display(ipd.Audio(audio, rate=sample_rate))
    print("Groud Truth (mel -> audio)")
    ipd.display(ipd.Audio(gt_audio, rate=sample_rate))
    print("Reconstruction (mel -> audio)")
    ipd.display(ipd.Audio(recon_audio, rate=sample_rate))
    plot_waveform(audio, sample_rate, title="Original waveform", ax=axs[0])
    plot_spectrogram(gt_m, title="GT", ax=axs[1])
    plot_spectrogram(recon_m, title="Recon", ax=axs[2])
    fig.tight_layout()

## Greedy

In [ ]:
class GreedyCTCDecoder(torch.nn.Module):
    def __init__(self, labels, blank=0):
        super().__init__()
        self.labels = labels
        self.blank = blank

    def forward(self, emission_batch):
        """Given a sequence emission over labels, get the best path
        Args:
          emission_batch (Tensor): Logit tensors. Shape `[batch, num_seq, num_label]`.

        Returns:
          List[str]: The resulting transcript
        """
        res = []
        for emission in emission_batch:
            indices = torch.argmax(emission, dim=-1)  # [num_seq,]
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            joined = " ".join([self.labels[i] for i in indices])
            res.append(joined.replace("|", " ").strip())
        return res

vocab = list(umm_model.tokenizer.get_vocab().keys())
greedy_decoder = GreedyCTCDecoder(vocab)

In [ ]:
from tqdm import tqdm
from recipes.datasets.mcc.mix import VocalWebDataModule

pl_datamodule = VocalWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
)

val_loaders = pl_datamodule.val_dataloader()
vocal_loader, speech_loader = val_loaders[0], val_loaders[1]
vocal_iter, speech_iter = iter(vocal_loader), iter(speech_loader)

def wer(it, verbose=True, max_i=1000):
    mean_wer = []
    for i, batch in tqdm(enumerate(it)):
        input_batch = {"audio": batch["audio"].to("cuda"), "text": batch["text"]}
        model_input = umm_model.prepare_feature(input_batch)
        model_output = umm_model.model(model_input)
        emission, recon_feature = model_output["logits"], model_output["recon_feature"]
        actual_transcript = batch["text"]
        greedy_transcript = greedy_decoder(emission)
        for j, (a, g) in enumerate(zip(actual_transcript, greedy_transcript)):
            greedy_wer = torchaudio.functional.edit_distance(a.lower(), g.lower()) / len(a)
            if verbose:
                print("=============================")
                print(f"Actual transcript: {a.lower()}")
                print(f"Greedy transcript: {g.lower()}")
                print(f"WER: {greedy_wer}")
                ipd.display(ipd.Audio(batch["audio"][j], rate=sample_rate))
            mean_wer.append(greedy_wer)
        if i >= max_i:
            break
    mean_wer = sum(mean_wer) / len(mean_wer)
    print(f"Mean WER: {mean_wer}")  

In [ ]:
max_i = 100000
verbose = False
wer(vocal_iter, verbose, max_i)

In [ ]:
max_i = 10000
verbose = False
wer(speech_iter, verbose, max_i)

# VQ Tokens

In [ ]:
from tqdm import tqdm
from recipes.datasets.mcc.mix import VocalWebDataModule

pl_datamodule = VocalWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
)

val_loaders = pl_datamodule.val_dataloader()
vocal_loader, speech_loader = val_loaders[0], val_loaders[1]
vocal_iter, speech_iter = iter(vocal_loader), iter(speech_loader)

def w2t(it, verbose=False, max_i=1000):
    tokens = []
    for i, batch in tqdm(enumerate(it)):
        wav = batch["audio"].to("cuda")
        vq_ids = umm_model.wav2token(wav)
        for j, (w, t) in enumerate(zip(wav, vq_ids)):
            if verbose:
                print("=============================")
                print(t)
                ipd.display(ipd.Audio(w.cpu(), rate=sample_rate))
            tokens.append(t.unique().cpu())
        if i >= max_i:
            break
    print(len(torch.cat(tokens, dim=0).unique()))

In [ ]:
max_i = 10000
verbose = False
w2t(vocal_iter, verbose, max_i)

In [ ]:
max_i = 10000
verbose = False
w2t(speech_iter, verbose, max_i)